# Async Python Basics

Every chapter 2 file uses the same four async pieces: `AsyncOpenAI`, `async def main()`, `await client.chat.completions.create(...)`, and `asyncio.run(main())`. This notebook teaches what those pieces mean, starting from zero. It uses only plain Python. The slow work is faked with `asyncio.sleep`, so there are **no AI calls, no API keys, and no internet** needed.

**How to use this notebook:** run the cells from top to bottom. Before each code cell, stop and **predict** what will print, especially the timings. Then run it and compare. Guessing wrong and then seeing why is how this material sticks. The whole notebook takes about 40 seconds to run. The exercises at the end are meant to be written by hand.

## Table of Contents

1. [Why Async Exists](#1-why-async-exists)
2. [Setup: Imports and a Clock](#2-setup-imports-and-a-clock)
3. [The Four Words You Need](#3-the-four-words-you-need)
4. [Jupyter vs Script: How Async Code Starts](#4-jupyter-vs-script-how-async-code-starts)
5. [Your First Coroutine](#5-your-first-coroutine)
6. [await Runs Things One After Another](#6-await-runs-things-one-after-another)
7. [gather: Running Things Together](#7-gather-running-things-together)
8. [create_task: Start Now, Collect Later](#8-create_task-start-now-collect-later)
9. [The Blocking Trap](#9-the-blocking-trap)
10. [Timeouts](#10-timeouts)
11. [When Things Fail](#11-when-things-fail)
12. [Limiting How Many Run at Once](#12-limiting-how-many-run-at-once)
13. [Handling Results as They Finish](#13-handling-results-as-they-finish)
14. [Putting It Together: A Fake Agent](#14-putting-it-together-a-fake-agent)
15. [From Notebook to Your Chapter 2 Scripts](#15-from-notebook-to-your-chapter-2-scripts)
16. [Gotchas Cheat Sheet](#16-gotchas-cheat-sheet)
17. [Practice Exercises (Hand-Write These)](#17-practice-exercises-hand-write-these)
18. [Exercise Solutions](#18-exercise-solutions)

<a id="1-why-async-exists"></a>

## 1. Why Async Exists

**In one sentence:** async lets one Python program do other useful work while it waits for slow things, instead of sitting idle.

### The Problem

An AI agent program spends most of its time **waiting**, not computing. A model API call takes 1 to 10 seconds to reply. A web search takes half a second. While that happens, your CPU does nothing. It is just waiting for data to come back over the network.

Normal Python code is called **synchronous** (or "sync") code. It waits patiently: start job 1, wait until it finishes, then start job 2, and so on. Say an agent needs 3 tools that each take 2 seconds, and the tools don't depend on each other. Sync code takes 6 seconds. It would be faster to send all three requests at once and wait for them together. That is what async does: about 2 seconds instead of 6.

### The Intuition: One Chai Stall Worker

Picture a chai stall with **one** worker and three orders: chai (1 minute to boil), toast (2 minutes), and an omelette (3 minutes).

A **sync worker** puts the chai on the stove and watches it until it boils. Only then do they put bread in the toaster, and they watch that too. Then they make the omelette. Total: 1 + 2 + 3 = 6 minutes, and most of that time they are just standing there.

An **async worker** puts the chai on. *While it boils*, they put bread in the toaster. *While both cook*, they start the omelette. Whenever something is done, they handle it. Total: about 3 minutes, which is how long the slowest item takes.

The key point is that it is still **one worker**. Async doesn't add more workers. It just stops the one worker from standing idle.

```text
SYNC: one thing at a time                            total = 6s
time:      0     1     2     3     4     5     6
chai       [=====]
toast            [===========]
omelette                     [=================]

ASYNC: start all, then wait for all together         total = 3s
time:      0     1     2     3
chai       [=====]
toast      [===========]
omelette   [=================]
```

### The Math

$$T_{\text{sync}} = t_1 + t_2 + \dots + t_n$$

$$T_{\text{async}} \approx \max(t_1, t_2, \dots, t_n)$$

Every symbol:

- $n$ is the number of jobs.
- $t_i$ is how long job number $i$ spends waiting (for example, $t_2$ is the toast's 2 seconds).
- $T_{\text{sync}}$ is the total time when jobs run one after another. You **add up** all the waits.
- $T_{\text{async}}$ is the total time when jobs wait together. You only pay for the **longest** wait.
- The $\approx$ ("roughly equal") is there because switching between jobs costs a tiny amount of time, a few microseconds, which is too small to notice here.

### Dry Run

```text
Given: t1 = 1s (chai), t2 = 2s (toast), t3 = 3s (omelette)

Sync:   T = 1 + 2 + 3          = 6s
Async:  T = max(1, 2, 3)       = 3s
Saved:  6 - 3                  = 3s   (2x faster)

Agent example: 10 tool calls, each takes 2s
Sync:   T = 2 + 2 + ... + 2    = 20s
Async:  T = max(2, 2, ..., 2)  = 2s   (10x faster)
```

### When Async Does NOT Help

Async only helps with **waiting**. The proper name for that is **I/O-bound work**. (I/O means input/output: talking to the network, the disk, or a database.) If your code is busy *computing*, like a big loop doing math, that is called **CPU-bound work**. In that case the one worker is never idle, so there is nothing to overlap. CPU-bound work needs real extra workers, such as multiple processes or a GPU. That's a separate topic.

*A note on two similar words:* overlapping many waiting jobs with one worker is called **concurrency**. Running jobs at the exact same instant on several CPU cores is called **parallelism**. Async is concurrency, not parallelism.

<a id="2-setup-imports-and-a-clock"></a>

## 2. Setup: Imports and a Clock

Run this cell first. It defines one helper, `log()`, which prints how many seconds have passed before every message. The timestamps let you *see* which jobs overlap. `reset_clock()` sets the clock back to zero at the start of each demo.

In [1]:
# ============================================================
# TOPIC: Async Python basics: coroutines, await, event loop, tasks
# MATH:  T_sync = t1 + t2 + ... + tn   vs   T_async ≈ max(t1, ..., tn)
# REF:   B05 ch02 from_scratch (groundwork for the AsyncOpenAI code)
# ============================================================

# --- Imports ---
import asyncio
import math
import sys
import time

# --- A shared clock so every log line shows WHEN it happened ---
CLOCK_START = time.perf_counter()


def reset_clock():
    """
    Restart the shared clock at zero. Call this at the start of every demo.

    Returns:
        None
    """
    global CLOCK_START
    CLOCK_START = time.perf_counter()


def log(message):
    """
    Print a message with the number of seconds since the last reset_clock().

    Args:
        message (str): what just happened
    Returns:
        None
    Note: time.perf_counter() is a precise stopwatch; later reading - earlier reading = seconds passed.
    """
    elapsed_seconds = time.perf_counter() - CLOCK_START
    print(f"  [t={elapsed_seconds:5.2f}s] {message}")


print("-" * 60)
print("SETUP")
print("-" * 60)
print(f"  Python version                          : {sys.version.split()[0]}")
print(f"  asyncio.timeout available (needs 3.11+) : {hasattr(asyncio, 'timeout')}")
print(f"  asyncio.TaskGroup available (3.11+)     : {hasattr(asyncio, 'TaskGroup')}")
reset_clock()
log("the clock works")

------------------------------------------------------------
SETUP
------------------------------------------------------------
  Python version                          : 3.12.10
  asyncio.timeout available (needs 3.11+) : True
  asyncio.TaskGroup available (3.11+)     : True
  [t= 0.00s] the clock works


<a id="3-the-four-words-you-need"></a>

## 3. The Four Words You Need

All of async Python is built on four ideas. Read these once now. Each one gets its own demo below.

- **Coroutine**: a function written with `async def`. It is a function that can **pause in the middle** and continue later. Calling it does *not* run it. You only get back a coroutine object, which is like a recipe card: it describes the meal but isn't the meal yet.
- **`await`**: means "pause me here until this is done, and let other jobs run in the meantime." You can only use it inside an `async def` function (and, as a special case, directly in a Jupyter cell).
- **Event loop**: the manager. It keeps a list of jobs and keeps asking "who is ready to run right now?" There is one event loop per program.
- **Task**: a coroutine that has been handed to the event loop to run in the background. `asyncio.create_task(...)` and `asyncio.gather(...)` both turn coroutines into tasks.

```text
                 ┌──────────────────────────────────┐
                 │            EVENT LOOP            │
                 │ "Who is ready to run right now?" │
                 └────────────────┬─────────────────┘
                                  │ gives the turn to ONE job at a time
          ┌───────────────────────┼───────────────────────┐
          ▼                       ▼                       ▼
    ┌───────────┐           ┌───────────┐           ┌───────────┐
    │   chai    │           │   toast   │           │ omelette  │
    │ (waiting) │           │  (ready)  │           │ (waiting) │
    └─────┬─────┘           └───────────┘           └───────────┘
          │
          └── when a job reaches an `await`, it hands the turn back to the loop
```

### The Golden Rule

**Async code only switches between jobs at an `await`.** Between two `await`s, your code runs without any interruption. This style is called **cooperative multitasking**: each job has to *choose* to give up its turn, and it does that by awaiting. A job that never awaits keeps the one worker to itself and nothing else runs. Section 9 shows how that goes wrong.

<a id="4-jupyter-vs-script-how-async-code-starts"></a>

## 4. Jupyter vs Script: How Async Code Starts

Async code needs a running event loop. Where that loop comes from depends on where your code runs:

- **In a `.py` script** (like your chapter 2 files), no loop exists until you start one. You do that once, at the very bottom, with `asyncio.run(main())`.
- **In Jupyter**, a loop is *already running*, because Jupyter itself uses one. So you just write `await something()` directly in a cell. Calling `asyncio.run()` here gives an error, because you can't start a second loop inside the one that is already running.

This cell shows both. The first attempt fails on purpose.

In [2]:
async def say_hello():
    """
    The smallest possible coroutine: it just returns a greeting.

    Returns:
        str: a greeting
    """
    return "hello from a coroutine"


print("-" * 60)
print("TRY 1: asyncio.run() inside Jupyter (the SCRIPT way)")
print("-" * 60)
hello_coroutine = say_hello()
try:
    asyncio.run(hello_coroutine)
    print("  asyncio.run worked, so no loop was running in this environment")
except RuntimeError as error:
    print(f"  RuntimeError: {error}")
    print("  -> Jupyter already runs an event loop, so it refuses to start a second one.")
    hello_coroutine.close()   # the coroutine never ran; close it so Python doesn't warn about it

print()
print("-" * 60)
print("TRY 2: plain top-level await (the NOTEBOOK way)")
print("-" * 60)
greeting = await say_hello()
print(f"  got: {greeting!r}")

------------------------------------------------------------
TRY 1: asyncio.run() inside Jupyter (the SCRIPT way)
------------------------------------------------------------
  RuntimeError: asyncio.run() cannot be called from a running event loop
  -> Jupyter already runs an event loop, so it refuses to start a second one.

------------------------------------------------------------
TRY 2: plain top-level await (the NOTEBOOK way)
------------------------------------------------------------
  got: 'hello from a coroutine'


**Takeaway:** the rest of this notebook uses top-level `await` in cells. When you move code into a `.py` file, wrap it in `async def main():` and add `asyncio.run(main())` at the bottom. Section 15 shows the exact template.

<a id="5-your-first-coroutine"></a>

## 5. Your First Coroutine

Here is the fake "slow job" the rest of the notebook uses. `asyncio.sleep(seconds)` stands in for any real wait, like an API call, a web request, or a file download. It's the async version of `time.sleep`. The difference is that it *gives the turn back* to the event loop while it waits.

In [3]:
async def make_order(item_name, cook_seconds):
    """
    Pretend to cook one item: wait without blocking, then return a message.

    Args:
        item_name (str): what we're cooking, e.g. "chai"
        cook_seconds (float): how long the fake cooking takes
    Returns:
        str: e.g. "chai is ready"
    Note: asyncio.sleep stands in for any real wait, like an API call or a web request.
    """
    log(f"start {item_name} ({cook_seconds}s)")
    await asyncio.sleep(cook_seconds)   # pause here; the event loop can run other jobs meanwhile
    log(f"done  {item_name}")
    return f"{item_name} is ready"


print("make_order() is defined. Nothing has run yet; defining a function never runs it.")

make_order() is defined. Nothing has run yet; defining a function never runs it.


**Predict first:** what happens if you call `make_order("chai", 1)` *without* `await`? Does it print "start chai"?

In [4]:
print("-" * 60)
print("Calling make_order() WITHOUT await")
print("-" * 60)
reset_clock()
not_started_yet = make_order("chai", 1)
print(f"  what we got back : {not_started_yet}")
print(f"  its type         : {type(not_started_yet).__name__}")
print("  Notice: no 'start chai' line was printed. The function body did NOT run.")
not_started_yet.close()   # tidy up so Python doesn't warn "coroutine was never awaited"

------------------------------------------------------------
Calling make_order() WITHOUT await
------------------------------------------------------------
  what we got back : <coroutine object make_order at 0x10a0f1a80>
  its type         : coroutine
  Notice: no 'start chai' line was printed. The function body did NOT run.


In [5]:
print("-" * 60)
print("Calling make_order() WITH await")
print("-" * 60)
reset_clock()
chai_message = await make_order("chai", 1)
print(f"  what we got back : {chai_message!r}")
print(f"  its type         : {type(chai_message).__name__}")

------------------------------------------------------------
Calling make_order() WITH await
------------------------------------------------------------
  [t= 0.00s] start chai (1s)
  [t= 1.00s] done  chai
  what we got back : 'chai is ready'
  its type         : str


### The Most Common Bug in Agent Code

If you forget `await`, you get a coroutine object where you expected a real value. Python doesn't complain at that line. It fails *later*, when you try to use the value:

In [6]:
print("-" * 60)
print("The forgot-await bug")
print("-" * 60)
forgot_await = make_order("chai", 1)   # oops: no await
try:
    print(forgot_await.upper())          # we expected a string here
except AttributeError as error:
    print(f"  AttributeError: {error}")
forgot_await.close()

------------------------------------------------------------
The forgot-await bug
------------------------------------------------------------
  AttributeError: 'coroutine' object has no attribute 'upper'


In chapter 2 this bug looks like this:

```python
response = client.chat.completions.create(model=MODEL_NAME, messages=messages)   # no await!
print(response.choices[0].message.content)
# AttributeError: 'coroutine' object has no attribute 'choices'
```

**Rule of thumb:** if the word *coroutine* shows up in an error message, you almost certainly forgot an `await`.

<a id="6-await-runs-things-one-after-another"></a>

## 6. await Runs Things One After Another

**Predict first:** this function awaits chai (1s), then toast (2s), then omelette (3s). How long does it take in total?

In [7]:
async def cook_one_by_one():
    """
    Await three orders one after another.

    Returns:
        list[str]: the three "... is ready" messages
    Note: T = 1 + 2 + 3 = 6s, because each await finishes before the next line starts.
    """
    chai_result = await make_order("chai", 1)
    toast_result = await make_order("toast", 2)
    omelette_result = await make_order("omelette", 3)
    return [chai_result, toast_result, omelette_result]


print("-" * 60)
print("One by one, with three separate awaits")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
one_by_one_results = await cook_one_by_one()
total_seconds = time.perf_counter() - start_time
print(f"\n  results    : {one_by_one_results}")
print(f"  total time : {total_seconds:.2f}s   (expected ≈ 1 + 2 + 3 = 6s)")

------------------------------------------------------------
One by one, with three separate awaits
------------------------------------------------------------
  [t= 0.00s] start chai (1s)
  [t= 1.00s] done  chai
  [t= 1.00s] start toast (2s)
  [t= 3.00s] done  toast
  [t= 3.00s] start omelette (3s)
  [t= 6.00s] done  omelette

  results    : ['chai is ready', 'toast is ready', 'omelette is ready']
  total time : 6.00s   (expected ≈ 1 + 2 + 3 = 6s)


**This surprises a lot of people.** Writing `async def` does *not* by itself make things run together. `await` means "wait for this to finish before moving to the next line," which is exactly what normal code does. Look at the log: toast only starts once chai is done.

So what is the point? While this function waits, the event loop is free to run *other* tasks. In a web server, for example, that means other users' requests. But to speed up *your own* jobs, you have to **start several of them before you wait**. That is what the next section does.

<a id="7-gather-running-things-together"></a>

## 7. gather: Running Things Together

`asyncio.gather(job1, job2, job3)` starts all the jobs at once and waits until **all** of them are done. It returns their results as a list.

**Predict first:** the same three orders, but gathered. How long does it take now?

In [8]:
async def cook_together():
    """
    Start all three orders at once and wait for all of them.

    Returns:
        list[str]: the three "... is ready" messages, in argument order
    Note: T ≈ max(1, 2, 3) = 3s.
    """
    all_results = await asyncio.gather(
        make_order("chai", 1),
        make_order("toast", 2),
        make_order("omelette", 3),
    )
    return all_results


print("-" * 60)
print("All together, with asyncio.gather")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
together_results = await cook_together()
total_seconds = time.perf_counter() - start_time
print(f"\n  results    : {together_results}")
print(f"  total time : {total_seconds:.2f}s   (expected ≈ max(1, 2, 3) = 3s)")

------------------------------------------------------------
All together, with asyncio.gather
------------------------------------------------------------
  [t= 0.00s] start chai (1s)
  [t= 0.00s] start toast (2s)
  [t= 0.00s] start omelette (3s)
  [t= 1.00s] done  chai
  [t= 2.00s] done  toast
  [t= 3.00s] done  omelette

  results    : ['chai is ready', 'toast is ready', 'omelette is ready']
  total time : 3.00s   (expected ≈ max(1, 2, 3) = 3s)


All three jobs started at `t=0.00`. That is the whole trick. Compare the timestamps with section 6.

### Results Come Back in the Order You Asked, Not the Order They Finished

Here the slowest job goes first. Watch the finish order in the log, then look at the results list:

In [9]:
print("-" * 60)
print("gather keeps argument order")
print("-" * 60)
reset_clock()
ordered_results = await asyncio.gather(
    make_order("slow soup", 0.9),
    make_order("quick chai", 0.3),
    make_order("medium toast", 0.6),
)
print(f"\n  finish order (see log) : quick chai -> medium toast -> slow soup")
print(f"  results list order     : {ordered_results}")
print("  -> slot 0 is always the first argument, whoever finished first.")

------------------------------------------------------------
gather keeps argument order
------------------------------------------------------------
  [t= 0.00s] start slow soup (0.9s)
  [t= 0.00s] start quick chai (0.3s)
  [t= 0.00s] start medium toast (0.6s)
  [t= 0.30s] done  quick chai
  [t= 0.60s] done  medium toast
  [t= 0.90s] done  slow soup

  finish order (see log) : quick chai -> medium toast -> slow soup
  results list order     : ['slow soup is ready', 'quick chai is ready', 'medium toast is ready']
  -> slot 0 is always the first argument, whoever finished first.


### Gathering a List of Jobs

In real code you often build the jobs in a loop. `gather` takes separate arguments, not one list, so you **unpack** the list with `*`. The `*` spreads the list's items out as separate arguments:

In [10]:
menu_cook_seconds = {"chai": 0.5, "toast": 0.5, "omelette": 0.5, "paratha": 0.5, "lassi": 0.5}

print("-" * 60)
print(f"Gathering {len(menu_cook_seconds)} jobs built in a loop")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
menu_jobs = [make_order(item_name, cook_seconds) for item_name, cook_seconds in menu_cook_seconds.items()]
menu_results = await asyncio.gather(*menu_jobs)   # *menu_jobs == menu_jobs[0], menu_jobs[1], ...
total_seconds = time.perf_counter() - start_time
print(f"\n  {len(menu_results)} results in {total_seconds:.2f}s   (5 jobs × 0.5s each, but ≈ 0.5s total)")

------------------------------------------------------------
Gathering 5 jobs built in a loop
------------------------------------------------------------
  [t= 0.00s] start chai (0.5s)
  [t= 0.00s] start toast (0.5s)
  [t= 0.00s] start omelette (0.5s)
  [t= 0.00s] start paratha (0.5s)
  [t= 0.00s] start lassi (0.5s)
  [t= 0.50s] done  chai
  [t= 0.50s] done  toast
  [t= 0.50s] done  omelette
  [t= 0.50s] done  paratha
  [t= 0.50s] done  lassi

  5 results in 0.50s   (5 jobs × 0.5s each, but ≈ 0.5s total)


<a id="8-create_task-start-now-collect-later"></a>

## 8. create_task: Start Now, Collect Later

`gather` means "start everything, then wait for everything." Sometimes you want to **start one job now**, do some other work, and **collect the result later**. `asyncio.create_task(coroutine)` hands the job to the event loop and gives you back a **Task** object right away. When you want the result, you `await` that task.

One detail to watch for: the task doesn't start the instant you create it. It starts the next time your code *gives up its turn* at an `await`. That's the golden rule from section 3 at work.

In [11]:
print("-" * 60)
print("create_task: start chai now, collect it later")
print("-" * 60)
reset_clock()
chai_task = asyncio.create_task(make_order("chai", 2))
log("chai task created (it starts at our first await, not right now)")
log("meanwhile, we do our own 1s job, like setting the table")
await asyncio.sleep(1)
log("our own job is done; now wait for the chai")
chai_result = await chai_task
log(f"got: {chai_result!r}")
print("\n  total ≈ max(2, 1) = 2s, not 2 + 1 = 3s")

------------------------------------------------------------
create_task: start chai now, collect it later
------------------------------------------------------------
  [t= 0.00s] chai task created (it starts at our first await, not right now)
  [t= 0.00s] meanwhile, we do our own 1s job, like setting the table
  [t= 0.00s] start chai (2s)
  [t= 1.00s] our own job is done; now wait for the chai
  [t= 2.00s] done  chai
  [t= 2.00s] got: 'chai is ready'

  total ≈ max(2, 1) = 2s, not 2 + 1 = 3s


Look at the log order: "start chai" was printed *after* our two log lines, even though the task was created first. It only started once we reached `await asyncio.sleep(1)`.

### Cancelling a Task

A task can be stopped with `task.cancel()`. Python then raises a special error called `asyncio.CancelledError` *inside* the job, at the `await` where it is currently paused. Here's a version of our job that reports when that happens. We'll use it for the rest of the notebook.

In [12]:
async def careful_job(job_name, work_seconds, should_fail=False):
    """
    A fake job that also reports when it gets cancelled or when it fails.

    Args:
        job_name (str): label for the log
        work_seconds (float): how long the fake work takes
        should_fail (bool): if True, raise ValueError after the wait
    Returns:
        str: e.g. "tool A ok"
    Raises:
        ValueError: if should_fail is True
    """
    log(f"start {job_name} ({work_seconds}s)")
    try:
        await asyncio.sleep(work_seconds)
    except asyncio.CancelledError:
        log(f"{job_name} was CANCELLED")
        raise   # always re-raise CancelledError so whoever cancelled us knows it worked
    if should_fail:
        log(f"{job_name} FAILED")
        raise ValueError(f"{job_name} broke")
    log(f"done  {job_name}")
    return f"{job_name} ok"


print("-" * 60)
print("Cancelling a running task")
print("-" * 60)
reset_clock()
soup_task = asyncio.create_task(careful_job("slow soup", 10))
await asyncio.sleep(0.5)
log("the customer left: cancel the soup")
soup_task.cancel()
try:
    await soup_task
except asyncio.CancelledError:
    log("confirmed: awaiting a cancelled task raises CancelledError")
print(f"\n  soup_task.cancelled() = {soup_task.cancelled()}   (we did not wait the full 10s)")

------------------------------------------------------------
Cancelling a running task
------------------------------------------------------------
  [t= 0.00s] start slow soup (10s)
  [t= 0.50s] the customer left: cancel the soup
  [t= 0.50s] slow soup was CANCELLED
  [t= 0.50s] confirmed: awaiting a cancelled task raises CancelledError

  soup_task.cancelled() = True   (we did not wait the full 10s)


**Gotchas with tasks:**

- **Keep a reference.** Store the task in a variable (or a list) until it's done. The event loop only holds a *weak* reference, which means it doesn't stop Python's garbage collector from deleting the task. A task nothing else points to can be thrown away halfway through.
- **In Jupyter, a task you never await keeps running** after its cell finishes. Its log lines can then show up under a *later* cell's output, which is confusing.
- **If you catch `CancelledError`, always `raise` it again**, as `careful_job` does. If you swallow it, cancellation silently stops working.

<a id="9-the-blocking-trap"></a>

## 9. The Blocking Trap

Remember the golden rule: jobs only switch at an `await`. So what happens when a job waits using a normal function that **doesn't** await, like `time.sleep`?

**Predict first:** three jobs of 1s each, gathered, but each one uses `time.sleep` instead of `asyncio.sleep`. Is the total 1s or 3s?

In [13]:
async def make_order_blocking(item_name, cook_seconds):
    """
    WRONG ON PURPOSE: an async function that waits with time.sleep.

    Args:
        item_name (str): what we're cooking
        cook_seconds (float): how long to block
    Returns:
        str: e.g. "chai is ready"
    """
    log(f"start {item_name}")
    time.sleep(cook_seconds)   # BLOCKS: no await, so the whole event loop is frozen here
    log(f"done  {item_name}")
    return f"{item_name} is ready"


print("-" * 60)
print("gather + time.sleep = no speed-up at all")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
await asyncio.gather(
    make_order_blocking("chai", 1),
    make_order_blocking("toast", 1),
    make_order_blocking("omelette", 1),
)
total_seconds = time.perf_counter() - start_time
print(f"\n  total time : {total_seconds:.2f}s   (3s, NOT 1s: each job froze the loop for the others)")

------------------------------------------------------------
gather + time.sleep = no speed-up at all
------------------------------------------------------------
  [t= 0.00s] start chai
  [t= 1.00s] done  chai
  [t= 1.00s] start toast
  [t= 2.01s] done  toast
  [t= 2.01s] start omelette
  [t= 3.01s] done  omelette

  total time : 3.01s   (3s, NOT 1s: each job froze the loop for the others)


Even with `gather`, the jobs ran one after another. `time.sleep` holds on to the only worker and never gives the turn back. The event loop can't start toast until chai's `time.sleep` is over.

This is a real risk in agent code, because many popular libraries **block**: `requests.get(...)`, the plain `OpenAI(...)` client (as opposed to `AsyncOpenAI`), and slow file reads. Use the async version when one exists:

| Blocks the loop (avoid inside `async def`) | Async-friendly replacement |
|---|---|
| `time.sleep(s)` | `await asyncio.sleep(s)` |
| `requests.get(url)` | `await httpx.AsyncClient().get(url)` |
| `OpenAI(...)` client | `AsyncOpenAI(...)` client |
| `Anthropic(...)` client | `AsyncAnthropic(...)` client |
| any slow normal function you can't change | `await asyncio.to_thread(func, arg1, arg2)` |

### The Escape Hatch: asyncio.to_thread

When there's no async version, `asyncio.to_thread` runs the blocking function in a separate helper **thread** (a second line of work that Python runs alongside the main one). You get back something you can `await`, and your event loop stays free:

In [14]:
def old_blocking_library_call(item_name, cook_seconds):
    """
    A normal (non-async) function that blocks, like requests.get() or the sync OpenAI client.

    Args:
        item_name (str): what we're cooking
        cook_seconds (float): how long it blocks
    Returns:
        str: e.g. "chai is ready"
    """
    log(f"start {item_name} (inside a helper thread)")
    time.sleep(cook_seconds)
    log(f"done  {item_name}")
    return f"{item_name} is ready"


print("-" * 60)
print("Same blocking function, wrapped in asyncio.to_thread")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
threaded_results = await asyncio.gather(
    asyncio.to_thread(old_blocking_library_call, "chai", 1),       # pass the function AND its args separately
    asyncio.to_thread(old_blocking_library_call, "toast", 1),
    asyncio.to_thread(old_blocking_library_call, "omelette", 1),
)
total_seconds = time.perf_counter() - start_time
print(f"\n  results    : {threaded_results}")
print(f"  total time : {total_seconds:.2f}s   (back to ≈ 1s)")

------------------------------------------------------------
Same blocking function, wrapped in asyncio.to_thread
------------------------------------------------------------
  [t= 0.01s] start chai (inside a helper thread)
  [t= 0.01s] start toast (inside a helper thread)
  [t= 0.01s] start omelette (inside a helper thread)
  [t= 1.02s] done  chai  [t= 1.02s] done  omelette
  [t= 1.02s] done  toast


  results    : ['chai is ready', 'toast is ready', 'omelette is ready']
  total time : 1.02s   (back to ≈ 1s)


**Gotcha:** write `asyncio.to_thread(func, arg)`, **not** `asyncio.to_thread(func(arg))`. The second version calls `func(arg)` immediately, right there in the event loop, so it blocks before `to_thread` ever sees it.

<a id="10-timeouts"></a>

## 10. Timeouts

**The problem:** an API can hang and never reply. An agent that waits forever is a broken agent. You need a way to say "give up after N seconds."

`asyncio.wait_for(job, timeout=seconds)` does exactly that. If the job isn't done in time, it **cancels** the job (as in section 8) and raises `TimeoutError`.

In [16]:
print("-" * 60)
print("wait_for: a 3s job with a 1s timeout")
print("-" * 60)
reset_clock()
try:
    slow_result = await asyncio.wait_for(careful_job("slow web search", 3), timeout=1.0)
    log(f"got: {slow_result}")
except TimeoutError:
    log("gave up after 1.0s -> TimeoutError")

------------------------------------------------------------
wait_for: a 3s job with a 1s timeout
------------------------------------------------------------
  [t= 0.00s] start slow web search (3s)
  [t= 1.00s] slow web search was CANCELLED
  [t= 1.00s] gave up after 1.0s -> TimeoutError


Notice the "CANCELLED" line: a timeout *is* a cancellation. The slow job didn't keep running in the background.

### A Time Limit for a Whole Block: asyncio.timeout

In Python 3.11 and later, `async with asyncio.timeout(seconds):` puts one shared time limit around several `await`s. Think of it as a total time allowance for a group of steps:

In [17]:
print("-" * 60)
print("asyncio.timeout: three 0.6s steps inside a 1.5s allowance")
print("-" * 60)
reset_clock()
try:
    async with asyncio.timeout(1.5):
        await careful_job("step 1", 0.6)
        await careful_job("step 2", 0.6)
        await careful_job("step 3", 0.6)   # would end at 1.8s, which is past the limit
    log("all steps finished in time")
except TimeoutError:
    log("the whole block ran out of its 1.5s allowance")

------------------------------------------------------------
asyncio.timeout: three 0.6s steps inside a 1.5s allowance
------------------------------------------------------------
  [t= 0.00s] start step 1 (0.6s)
  [t= 0.60s] done  step 1
  [t= 0.60s] start step 2 (0.6s)
  [t= 1.20s] done  step 2
  [t= 1.20s] start step 3 (0.6s)
  [t= 1.50s] step 3 was CANCELLED
  [t= 1.50s] the whole block ran out of its 1.5s allowance


<a id="11-when-things-fail"></a>

## 11. When Things Fail

When you run several jobs together and one of them raises an error, what happens to the others? There are three common patterns, and they behave differently.

### Pattern A: Plain gather (the Default)

The first error comes straight out of your `await gather(...)`. You lose all the results, including the ones that succeeded. And the other jobs **are not stopped**. They keep running in the background.

In [18]:
print("-" * 60)
print("Pattern A: plain gather, tool B fails")
print("-" * 60)
reset_clock()
try:
    pattern_a_results = await asyncio.gather(
        careful_job("tool A", 1.0),
        careful_job("tool B", 0.5, should_fail=True),
        careful_job("tool C", 1.0),
    )
except ValueError as error:
    log(f"gather raised {error!r}; results from A and C are LOST")
log("wait 0.7s more to see what A and C do...")
await asyncio.sleep(0.7)
log("see above: A and C kept running and finished anyway")

------------------------------------------------------------
Pattern A: plain gather, tool B fails
------------------------------------------------------------
  [t= 0.00s] start tool A (1.0s)
  [t= 0.00s] start tool B (0.5s)
  [t= 0.00s] start tool C (1.0s)
  [t= 0.50s] tool B FAILED
  [t= 0.50s] gather raised ValueError('tool B broke'); results from A and C are LOST
  [t= 0.50s] wait 0.7s more to see what A and C do...
  [t= 1.00s] done  tool A
  [t= 1.00s] done  tool C
  [t= 1.20s] see above: A and C kept running and finished anyway


### Pattern B: gather(..., return_exceptions=True)

Now errors are **returned in the results list** instead of being raised. Every job gets a slot, and a failed job's slot holds the error object. This is the usual choice for agents, where one broken tool shouldn't throw away the other tools' answers.

In [19]:
print("-" * 60)
print("Pattern B: return_exceptions=True")
print("-" * 60)
reset_clock()
pattern_b_results = await asyncio.gather(
    careful_job("tool A", 1.0),
    careful_job("tool B", 0.5, should_fail=True),
    careful_job("tool C", 1.0),
    return_exceptions=True,
)
print()
for job_name, job_result in zip(["tool A", "tool B", "tool C"], pattern_b_results):
    if isinstance(job_result, Exception):
        print(f"  {job_name}: FAILED -> {job_result!r}")
    else:
        print(f"  {job_name}: ok     -> {job_result!r}")

------------------------------------------------------------
Pattern B: return_exceptions=True
------------------------------------------------------------
  [t= 0.00s] start tool A (1.0s)
  [t= 0.00s] start tool B (0.5s)
  [t= 0.00s] start tool C (1.0s)
  [t= 0.50s] tool B FAILED
  [t= 1.00s] done  tool A
  [t= 1.00s] done  tool C

  tool A: ok     -> 'tool A ok'
  tool B: FAILED -> ValueError('tool B broke')
  tool C: ok     -> 'tool C ok'


### Pattern C: TaskGroup (Python 3.11+)

`asyncio.TaskGroup` is the newer and stricter option. If any job fails, it **cancels all the others** and raises the errors together as an `ExceptionGroup`, which you catch with the special `except*` syntax. Use it when the jobs only make sense together: if one fails, there's no point finishing the rest.

In [20]:
async def run_with_task_group():
    """
    Run three jobs in a TaskGroup where one fails.

    Returns:
        None
    Note: on the first failure, TaskGroup cancels the rest, then raises an ExceptionGroup.
    """
    try:
        async with asyncio.TaskGroup() as task_group:
            task_group.create_task(careful_job("tool A", 1.0))
            task_group.create_task(careful_job("tool B", 0.5, should_fail=True))
            task_group.create_task(careful_job("tool C", 1.0))
        log("all jobs finished")   # never reached in this demo
    except* ValueError as error_group:
        for error in error_group.exceptions:
            log(f"TaskGroup raised {error!r}")


print("-" * 60)
print("Pattern C: TaskGroup, tool B fails")
print("-" * 60)
reset_clock()
await run_with_task_group()

------------------------------------------------------------
Pattern C: TaskGroup, tool B fails
------------------------------------------------------------
  [t= 0.00s] start tool A (1.0s)
  [t= 0.00s] start tool B (0.5s)
  [t= 0.00s] start tool C (1.0s)
  [t= 0.50s] tool B FAILED
  [t= 0.50s] tool A was CANCELLED
  [t= 0.50s] tool C was CANCELLED
  [t= 0.50s] TaskGroup raised ValueError('tool B broke')


| Pattern | On an error, the other jobs... | You get back... | Use when |
|---|---|---|---|
| `gather(...)` | keep running (in the background) | only the first error | quick scripts where any error means "stop" |
| `gather(..., return_exceptions=True)` | keep running, results kept | a list mixing results and errors | independent tools; partial answers are useful |
| `TaskGroup` | get **cancelled** | an `ExceptionGroup` | jobs that only make sense together |

<a id="12-limiting-how-many-run-at-once"></a>

## 12. Limiting How Many Run at Once

**The problem:** `gather` on 50 API calls fires all 50 at the same moment. Most providers allow only a limited number of requests at a time and reply with a "429 Too Many Requests" error once you go over. You want concurrency, but capped.

**The intuition:** a restaurant with 2 tables. Customers wait at the door. As soon as a table frees up, the next person gets seated.

`asyncio.Semaphore(k)` is that restaurant. It holds $k$ passes. `async with semaphore:` takes a pass (or waits at the door if none are left) and hands it back automatically when the block ends, even if an error happens inside.

### The Math

$$T \approx \left\lceil \frac{n}{k} \right\rceil \times t$$

- $n$ is the number of jobs, $k$ is the limit (number of passes), and $t$ is the time each job takes.
- $\lceil \cdot \rceil$ means **round up** to the next whole number. A half-full batch still takes a full round.

### Dry Run

```text
Given: n = 5 customers, k = 2 tables, t = 1s each

Rounds needed:  5 / 2 = 2.5  -> round up -> 3
Total time:     3 × 1s = 3s

time:     0         1         2         3
table 1   [Asha    ][Chitra  ][Esha    ]
table 2   [Bikram  ][Dev     ]
```

In [21]:
TABLE_LIMIT = 2
restaurant_tables = asyncio.Semaphore(TABLE_LIMIT)


async def limited_order(customer_name, eat_seconds):
    """
    Wait for a free table (a semaphore pass), eat, then give the table back.

    Args:
        customer_name (str): who is eating
        eat_seconds (float): how long they sit at the table
    Returns:
        str: e.g. "Asha served"
    Note: at most TABLE_LIMIT customers are ever inside the `async with` block at the same time.
    """
    log(f"{customer_name} waits at the door")
    async with restaurant_tables:            # take a pass, or wait here until one is free
        log(f"{customer_name} SEATED")
        await asyncio.sleep(eat_seconds)
        log(f"{customer_name} leaves")
    return f"{customer_name} served"         # the pass was handed back when the block ended


customer_names = ["Asha", "Bikram", "Chitra", "Dev", "Esha"]
expected_seconds = math.ceil(len(customer_names) / TABLE_LIMIT) * 1.0

print("-" * 60)
print(f"{len(customer_names)} customers, {TABLE_LIMIT} tables, 1s each")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
await asyncio.gather(*(limited_order(name, 1.0) for name in customer_names))
total_seconds = time.perf_counter() - start_time
print(f"\n  total time : {total_seconds:.2f}s   (expected ≈ ceil(5/2) × 1 = {expected_seconds:.0f}s)")

------------------------------------------------------------
5 customers, 2 tables, 1s each
------------------------------------------------------------
  [t= 0.00s] Asha waits at the door
  [t= 0.00s] Asha SEATED
  [t= 0.00s] Bikram waits at the door
  [t= 0.00s] Bikram SEATED
  [t= 0.00s] Chitra waits at the door
  [t= 0.00s] Dev waits at the door
  [t= 0.00s] Esha waits at the door
  [t= 1.00s] Asha leaves
  [t= 1.00s] Bikram leaves
  [t= 1.00s] Chitra SEATED
  [t= 1.00s] Dev SEATED
  [t= 2.00s] Chitra leaves
  [t= 2.00s] Dev leaves
  [t= 2.00s] Esha SEATED
  [t= 3.00s] Esha leaves

  total time : 3.00s   (expected ≈ ceil(5/2) × 1 = 3s)


<a id="13-handling-results-as-they-finish"></a>

## 13. Handling Results as They Finish

`gather` hands you *everything at the end*. Sometimes you want to handle each result **the moment it's ready**. For example, you might show the fastest search result first, or update a progress bar. `asyncio.as_completed(jobs)` gives the jobs back in **finish order**:

In [22]:
print("-" * 60)
print("as_completed: handle each result as soon as it lands")
print("-" * 60)
reset_clock()
search_jobs = [
    make_order("slow search", 0.9),
    make_order("fast search", 0.3),
    make_order("medium search", 0.6),
]
for finish_position, next_finished_job in enumerate(asyncio.as_completed(search_jobs), start=1):
    search_result = await next_finished_job
    log(f"#{finish_position} to finish -> handled {search_result!r} right away")

------------------------------------------------------------
as_completed: handle each result as soon as it lands
------------------------------------------------------------
  [t= 0.00s] start medium search (0.6s)
  [t= 0.00s] start fast search (0.3s)
  [t= 0.00s] start slow search (0.9s)
  [t= 0.30s] done  fast search
  [t= 0.30s] #1 to finish -> handled 'fast search is ready' right away
  [t= 0.60s] done  medium search
  [t= 0.60s] #2 to finish -> handled 'medium search is ready' right away
  [t= 0.90s] done  slow search
  [t= 0.90s] #3 to finish -> handled 'slow search is ready' right away


| Tool | Order of results | You get results |
|---|---|---|
| `gather` | the order you passed them in | all at once, at the end |
| `as_completed` | the order they finish | one by one, as each is ready |

<a id="14-putting-it-together-a-fake-agent"></a>

## 14. Putting It Together: A Fake Agent

Here is everything combined into the shape a real tool-using agent has. There is no model in it, just fakes with `asyncio.sleep`:

1. Ask the "model" for a plan (one awaited call).
2. Run 4 "tools" **concurrently**, with at most 3 at once (semaphore), a 1.5s timeout each (`wait_for`), and errors turned into readable text so one broken tool can't crash the agent.
3. Ask the "model" for a final answer using the tool results.

```text
 USER question
      │
      ▼
 [fake LLM: plan] ─────────────── 1.0s
      │
      ▼  gather (max 3 at once, 1.5s timeout each)
 ┌──────────┬──────────┬───────────┬───────────────┐
 │ weather  │  news    │  stocks   │ slow search   │
 │  0.8s ✓  │  1.2s ✓  │ 0.4s FAIL │ waits a slot, │
 │          │          │           │ then TIMEOUT  │
 └──────────┴──────────┴───────────┴───────────────┘
      │
      ▼
 [fake LLM: final answer] ─────── 1.0s
```

In [23]:
TOOL_TIMEOUT_SECONDS = 1.5
MAX_TOOLS_AT_ONCE = 3
tool_slots = asyncio.Semaphore(MAX_TOOLS_AT_ONCE)


async def fake_llm_call(prompt, thinking_seconds=1.0):
    """
    Pretend to be a model API call: wait, then return a canned reply.

    Args:
        prompt (str): what we "send" to the model
        thinking_seconds (float): fake network + generation time
    Returns:
        str: a fake reply
    Note: in real code this line is `await client.chat.completions.create(...)`.
    """
    log(f"LLM  <- {prompt!r}")
    await asyncio.sleep(thinking_seconds)
    fake_reply = f"(fake reply to: {prompt})"
    log(f"LLM  -> {fake_reply!r}")
    return fake_reply


async def weather_tool(city):
    """Fake weather API. Args: city (str). Returns: str."""
    log(f"tool weather started for {city}")
    await asyncio.sleep(0.8)
    return f"{city}: 31°C, humid"


async def news_tool(topic):
    """Fake news API. Args: topic (str). Returns: str."""
    log(f"tool news started for {topic}")
    await asyncio.sleep(1.2)
    return f"3 headlines about {topic}"


async def stock_tool(symbol):
    """Fake stock API that is down today. Args: symbol (str). Raises: ConnectionError."""
    log(f"tool stocks started for {symbol}")
    await asyncio.sleep(0.4)
    raise ConnectionError("stock API is down")


async def slow_search_tool(query):
    """Fake search API that hangs. Args: query (str). Returns: str (never, in practice)."""
    log(f"tool search started for {query!r}")
    await asyncio.sleep(10)
    return "search results"


async def run_tool_safely(tool_name, tool_coroutine):
    """
    Run one tool with a slot limit and a timeout, and turn any failure into text.

    Args:
        tool_name (str): label for the result
        tool_coroutine (coroutine): the tool call, created but not yet awaited
    Returns:
        str: the tool's answer, or a readable failure message (never raises)
    """
    async with tool_slots:
        try:
            tool_answer = await asyncio.wait_for(tool_coroutine, timeout=TOOL_TIMEOUT_SECONDS)
            log(f"tool {tool_name} done")
            return f"{tool_name}: {tool_answer}"
        except TimeoutError:
            log(f"tool {tool_name} TIMED OUT")
            return f"{tool_name}: gave up after {TOOL_TIMEOUT_SECONDS}s"
        except Exception as error:
            log(f"tool {tool_name} FAILED")
            return f"{tool_name}: failed ({error})"


async def run_fake_agent(user_question):
    """
    A pretend agent: plan with the fake model, run 4 tools concurrently, then answer.

    Args:
        user_question (str): what the user asked
    Returns:
        str: the fake final answer
    """
    log(f"USER : {user_question}")
    log("STEP 1: ask the model for a plan")
    plan = await fake_llm_call(f"make a plan for: {user_question}")

    log(f"STEP 2: run 4 tools concurrently (max {MAX_TOOLS_AT_ONCE} at once, {TOOL_TIMEOUT_SECONDS}s timeout each)")
    tool_results = await asyncio.gather(
        run_tool_safely("weather", weather_tool("Guwahati")),
        run_tool_safely("news", news_tool("monsoon")),
        run_tool_safely("stocks", stock_tool("TEA")),
        run_tool_safely("search", slow_search_tool("best chai near me")),
    )
    for tool_line in tool_results:
        log(f"  result -> {tool_line}")

    log("STEP 3: ask the model for the final answer")
    final_answer = await fake_llm_call(f"answer using the plan ({len(plan)} chars) + {len(tool_results)} tool results")
    return final_answer


print("-" * 60)
print("Running the fake agent")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
agent_answer = await run_fake_agent("Should I carry an umbrella in Guwahati today?")
total_seconds = time.perf_counter() - start_time

sequential_seconds = 1.0 + (0.8 + 1.2 + 0.4 + TOOL_TIMEOUT_SECONDS) + 1.0
print(f"\n  final answer : {agent_answer}")
print(f"  total time   : {total_seconds:.2f}s")
print(f"  if everything ran one by one: {sequential_seconds:.1f}s")

------------------------------------------------------------
Running the fake agent
------------------------------------------------------------
  [t= 0.00s] USER : Should I carry an umbrella in Guwahati today?
  [t= 0.00s] STEP 1: ask the model for a plan
  [t= 0.00s] LLM  <- 'make a plan for: Should I carry an umbrella in Guwahati today?'
  [t= 1.00s] LLM  -> '(fake reply to: make a plan for: Should I carry an umbrella in Guwahati today?)'
  [t= 1.00s] STEP 2: run 4 tools concurrently (max 3 at once, 1.5s timeout each)
  [t= 1.00s] tool weather started for Guwahati
  [t= 1.00s] tool news started for monsoon
  [t= 1.00s] tool stocks started for TEA
  [t= 1.40s] tool stocks FAILED
  [t= 1.40s] tool search started for 'best chai near me'
  [t= 1.80s] tool weather done
  [t= 2.20s] tool news done
  [t= 2.90s] tool search TIMED OUT
  [t= 2.90s]   result -> weather: Guwahati: 31°C, humid
  [t= 2.90s]   result -> news: 3 headlines about monsoon
  [t= 2.90s]   result -> stocks: failed (stock

### Read the Timeline

- `t=1.0`: the plan is done. Weather, news, and stocks start together. Search **waits for a free slot**, because only 3 tools can run at once.
- `t≈1.4`: stocks fails. Its slot frees up, so search starts.
- `t≈1.8` and `t≈2.2`: weather and news finish.
- `t≈2.9`: search hits its 1.5s timeout (it started at 1.4) and gets cancelled.
- The final model call runs from about 2.9s to 3.9s.

The stock tool broke and the search tool hung, but the agent still finished with a full answer. That's the result of returning error text instead of raising, and of using timeouts.

### How This Maps to Your Chapter 2 Code

| In this notebook | In your chapter 2 code |
|---|---|
| `await fake_llm_call(prompt)` | `await client.chat.completions.create(model=..., messages=...)` |
| `await weather_tool(city)` | your own tool functions (file 07 onwards) |
| `await asyncio.sleep(...)` | the real network wait inside the client |
| `asyncio.gather(...)` | running several tool calls the model asked for, all at once |
| `asyncio.wait_for(..., timeout=...)` | not letting a hung API freeze the agent |
| `asyncio.Semaphore(k)` | staying under the provider's rate limit |

<a id="15-from-notebook-to-your-chapter-2-scripts"></a>

## 15. From Notebook to Your Chapter 2 Scripts

In a `.py` file there's no loop already running, so every script uses the same template:

```python
import asyncio

from openai import AsyncOpenAI

client = AsyncOpenAI(base_url="...", api_key="...")   # creating the client does NOT need await


async def main():
    """Everything that awaits lives in here (or in async functions main calls)."""
    response = await client.chat.completions.create(model=MODEL_NAME, messages=messages)
    print(response.choices[0].message.content)


if __name__ == "__main__":      # only run when executed directly, not when imported
    asyncio.run(main())         # start the event loop ONCE, run main() to the end, close the loop
```

### "Async Is Contagious"

Any function that uses `await` must itself be `async def`. Its caller must then `await` it, so the caller must be `async def` too, and so on, all the way up to `main()`. At the very top, `asyncio.run()` is the one bridge from normal code into async code:

```text
asyncio.run(main())                <- the ONE place normal code enters async code
   └── async def main()
          └── await run_agent()            must be async: it awaits
                 └── await call_model()    must be async: it awaits
                        └── await client.chat.completions.create(...)
```

Call `asyncio.run()` **once per program**, at the bottom. Never call it inside an `async def`. You'd get the same "cannot be called from a running event loop" error you saw in section 4.

### Same Pattern, Both Providers

The async shape is identical whichever SDK you use. Only the method name and the response shape change:

```python
# OpenAI-compatible (Gemini / Ollama), which is what chapter 2 uses
from openai import AsyncOpenAI
client = AsyncOpenAI(base_url="...", api_key="...")
response = await client.chat.completions.create(model=MODEL_NAME, messages=messages)
reply_text = response.choices[0].message.content

# Anthropic: same async pattern, different method and response shape
from anthropic import AsyncAnthropic
client = AsyncAnthropic()
response = await client.messages.create(model=MODEL_NAME, max_tokens=1024, messages=messages)
reply_text = response.content[0].text
```

Both clients also take their own `timeout=` and `max_retries=` settings when you create them. For a single call, those settings are usually enough, and you don't need to wrap it in `asyncio.wait_for` yourself.

<a id="16-gotchas-cheat-sheet"></a>

## 16. Gotchas Cheat Sheet

1. **Forgot `await`** → you get a `coroutine` object instead of a value, and the error mentions *coroutine*.
2. **`await` one by one is still sequential.** To overlap jobs, start them first (`gather` / `create_task`), then wait.
3. **`time.sleep`, `requests`, or the sync `OpenAI` client inside `async def`** freeze the whole loop. Use the async version or `asyncio.to_thread`.
4. **`asyncio.run()` in Jupyter fails.** Use top-level `await` in notebooks, and `asyncio.run(main())` in scripts.
5. **`asyncio.run()` belongs at the bottom of the script, once.** Never call it inside an async function.
6. **Plain `gather` + one error** → you lose all results and the other jobs keep running. Use `return_exceptions=True` or `TaskGroup`.
7. **Keep a reference to every task** until it's done.
8. **Catching `CancelledError`? Re-raise it.**
9. **`to_thread(func, arg)`**, not `to_thread(func(arg))`.
10. **Async only speeds up waiting** (I/O-bound work), never heavy computing (CPU-bound work).

### Quick Reference

| I want to... | Use |
|---|---|
| define a pausable function | `async def f(): ...` |
| wait for one job | `result = await f()` |
| run many, wait for all | `results = await asyncio.gather(f(), g(), h())` |
| start now, collect later | `task = asyncio.create_task(f())` ... `await task` |
| stop a running task | `task.cancel()` |
| give up after N seconds | `await asyncio.wait_for(f(), timeout=N)` |
| one time limit for a whole block | `async with asyncio.timeout(N): ...` |
| keep going when some jobs fail | `gather(..., return_exceptions=True)` |
| all or nothing | `async with asyncio.TaskGroup() as tg: tg.create_task(f())` |
| at most k at once | `sem = asyncio.Semaphore(k)` + `async with sem: ...` |
| handle results as they finish | `for job in asyncio.as_completed(jobs): r = await job` |
| call blocking code safely | `await asyncio.to_thread(func, arg)` |
| start async code from a script | `asyncio.run(main())` |

<a id="17-practice-exercises-hand-write-these"></a>

## 17. Practice Exercises (Hand-Write These)

Write each answer **by hand**, with AI autocomplete turned off. Each exercise has a stub cell to fill in and a check cell to run afterwards. The check cell prints "not done yet" until you fill in the stub, so running the whole notebook never breaks.

When you're stuck, go through these steps in order: re-read the section named in the hint → read the error message carefully → look up the docs → only then scroll to section 18.

### Exercise 1: Breakfast Together

Write `cook(item, seconds)` so it logs a start line, waits *without blocking*, logs a done line, and returns `f"{item} ready"`. Then write `make_breakfast()` so it cooks paratha (1.0s), omelette (0.8s), and chai (0.5s) **at the same time** and returns the list of results in that order.

*Hint: sections 5 and 7.*

In [38]:
async def cook(item, seconds):
    """
    Log start, wait `seconds` without blocking, log done, return f"{item} ready".

    Args:
        item (str): what to cook
        seconds (float): how long it takes
    Returns:
        str: f"{item} ready"
    """
    # TODO: your code here
    print(f"started cooking {item} for {seconds} secs")
    await asyncio.sleep(seconds)
    print(f"done cooking {item}")
    return f"{item} ready"
    


async def make_breakfast():
    """
    Cook paratha (1.0s), omelette (0.8s) and chai (0.5s) at the same time.

    Returns:
        list[str]: ["paratha ready", "omelette ready", "chai ready"]
    """
    # TODO: your code here
    res = await asyncio.gather(
        cook("paratha", 1.0),
        cook("omelette", 0.8),
        cook("chai", 0.5)
    )

    return res

In [39]:
# --- Check: Exercise 1 ---
print("-" * 60)
print("Checking Exercise 1")
print("-" * 60)
reset_clock()
start_time = time.perf_counter()
breakfast_result = await make_breakfast()
total_seconds = time.perf_counter() - start_time

if breakfast_result is None:
    print("  Exercise 1 not done yet: fill in cook() and make_breakfast() above.")
else:
    assert list(breakfast_result) == ["paratha ready", "omelette ready", "chai ready"], \
        f"wrong results or wrong order: {breakfast_result}"
    assert total_seconds < 1.3, \
        f"took {total_seconds:.2f}s; expected ≈ max(1.0, 0.8, 0.5) = 1.0s. Are you awaiting one by one?"
    print(f"  results : {breakfast_result}")
    print(f"  time    : {total_seconds:.2f}s")
    print("  Exercise 1 PASSED")

------------------------------------------------------------
Checking Exercise 1
------------------------------------------------------------
started cooking paratha for 1.0 secs
started cooking omelette for 0.8 secs
started cooking chai for 0.5 secs
done cooking chai
done cooking omelette
done cooking paratha
  results : ['paratha ready', 'omelette ready', 'chai ready']
  time    : 1.00s
  Exercise 1 PASSED


### Exercise 2: Unfreeze the Loop

`legacy_fetch_price` is an old blocking function that you are **not allowed to change**. Each call takes 1 second. Write `fetch_all_prices(item_names)` so it fetches all prices **concurrently** and returns them in the same order as `item_names`. Three items should take about 1s, not 3s.

*Hint: section 9.*

In [ ]:
def legacy_fetch_price(item_name):
    """
    Old blocking price lookup. Do NOT edit this function.

    Args:
        item_name (str): one of "tea", "coffee", "lassi"
    Returns:
        int: price in rupees
    """
    time.sleep(1.0)
    return {"tea": 20, "coffee": 35, "lassi": 40}[item_name]


async def fetch_all_prices(item_names):
    """
    Fetch every price concurrently without freezing the event loop.

    Args:
        item_names (list[str]): items to look up
    Returns:
        list[int]: prices in the same order as item_names
    """
    # TODO: your code here
    # use asyncio.to_thread for legacy non async function
    item_list = [asyncio.to_thread(legacy_fetch_price, item) for item in item_names]
    res = await asyncio.gather(*item_list)

    return res

In [47]:
# --- Check: Exercise 2 ---
print("-" * 60)
print("Checking Exercise 2")
print("-" * 60)
start_time = time.perf_counter()
price_result = await fetch_all_prices(["tea", "coffee", "lassi"])
total_seconds = time.perf_counter() - start_time

if price_result is None:
    print("  Exercise 2 not done yet: fill in fetch_all_prices() above.")
else:
    assert list(price_result) == [20, 35, 40], f"wrong prices or wrong order: {price_result}"
    assert total_seconds < 1.5, \
        f"took {total_seconds:.2f}s; expected ≈ 1s. Is something still blocking the loop?"
    print(f"  prices : {price_result}")
    print(f"  time   : {total_seconds:.2f}s")
    print("  Exercise 2 PASSED")

------------------------------------------------------------
Checking Exercise 2
------------------------------------------------------------
  prices : [20, 35, 40]
  time   : 1.01s
  Exercise 2 PASSED


### Exercise 3: The Impatient Caller

Write `call_with_deadline(job_coroutine, deadline_seconds, fallback_value)`. It should await the job, but give up after `deadline_seconds` and return `fallback_value` instead. A job that finishes in time returns its real result.

*Hint: section 10.*

In [52]:
async def call_with_deadline(job_coroutine, deadline_seconds, fallback_value):
    """
    Await a job, but return fallback_value if it takes longer than deadline_seconds.

    Args:
        job_coroutine (coroutine): the job, created but not yet awaited
        deadline_seconds (float): the most we're willing to wait
        fallback_value: what to return on timeout
    Returns:
        the job's result, or fallback_value on timeout
    """
    # TODO: your code here
    try:
        res = await asyncio.wait_for(job_coroutine, deadline_seconds)
        return res
    # specifically for timeout use TimeoutError
    except TimeoutError:
        return fallback_value

In [53]:
# --- Check: Exercise 3 ---
print("-" * 60)
print("Checking Exercise 3")
print("-" * 60)
fast_job = make_order("fast tea", 0.2)
slow_job = make_order("slow biryani", 2.0)
reset_clock()
start_time = time.perf_counter()
fast_result = await call_with_deadline(fast_job, 1.0, "too slow")
slow_result = await call_with_deadline(slow_job, 0.5, "too slow")
total_seconds = time.perf_counter() - start_time
fast_job.close()   # no-op if it already ran; stops a warning if the stub is still empty
slow_job.close()

if fast_result is None and slow_result is None:
    print("  Exercise 3 not done yet: fill in call_with_deadline() above.")
else:
    assert fast_result == "fast tea is ready", f"fast job should return its real result, got {fast_result!r}"
    assert slow_result == "too slow", f"slow job should return the fallback, got {slow_result!r}"
    assert total_seconds < 1.2, \
        f"took {total_seconds:.2f}s; expected ≈ 0.2 + 0.5 = 0.7s. Did you really give up at the deadline?"
    print(f"  fast : {fast_result!r}")
    print(f"  slow : {slow_result!r}")
    print(f"  time : {total_seconds:.2f}s")
    print("  Exercise 3 PASSED")

------------------------------------------------------------
Checking Exercise 3
------------------------------------------------------------
  [t= 0.00s] start fast tea (0.2s)
  [t= 0.20s] done  fast tea
  [t= 0.20s] start slow biryani (2.0s)
  fast : 'fast tea is ready'
  slow : 'too slow'
  time : 0.70s
  Exercise 3 PASSED


<a id="18-exercise-solutions"></a>

## 18. Exercise Solutions

**Try first.** These solutions are markdown on purpose, not runnable cells, so you have to type them yourself. If you did look, close this section, wait a bit, and write the answer again from memory. That second attempt is where the learning happens.

### Solution 1

```python
async def cook(item, seconds):
    log(f"start {item}")
    await asyncio.sleep(seconds)          # asyncio.sleep, NOT time.sleep
    log(f"done  {item}")
    return f"{item} ready"


async def make_breakfast():
    return await asyncio.gather(          # start all three first, then wait once
        cook("paratha", 1.0),
        cook("omelette", 0.8),
        cook("chai", 0.5),
    )
```

### Solution 2

```python
async def fetch_all_prices(item_names):
    return await asyncio.gather(
        *(asyncio.to_thread(legacy_fetch_price, item_name) for item_name in item_names)
    )
```

The `*` unpacks the generator into separate arguments for `gather`. Also note that the function and its argument are passed **separately** to `to_thread`.

### Solution 3

```python
async def call_with_deadline(job_coroutine, deadline_seconds, fallback_value):
    try:
        return await asyncio.wait_for(job_coroutine, timeout=deadline_seconds)
    except TimeoutError:
        return fallback_value
```

### Where This Goes Next

You now know enough async to read every chapter 2 file. In file 01, `await client.chat.completions.create(...)` is a coroutine exactly like `make_order`. From file 07 onwards, tool calls follow the same pattern as the fake agent in section 14. Later chapters run several tools concurrently, which is `gather` with error handling, the same thing you just built.